# ESML v2: historical inference and named lake data
This notebook renders an inference request against an explicit model version. Supply existing customer resource settings and review the chosen model. No `0`, `latest`, active pointer or implicit Dev-to-Prod promotion is used.

In [ ]:
from pathlib import Path
from uuid import uuid4
from azure_esml import ESMLProject, PipelineRequest, PipelineType
project = ESMLProject.from_json(Path('lake_settings.json'))
request = PipelineRequest(data_date_utc='2026-09-12', model_version='7', run_id='inference-' + uuid4().hex, allow_reuse=False)
plan = project.create_pipeline(PipelineType.IN_2_GOLD_INFERENCE, request, output=Path('generated') / request.run_id)
print(plan.document['experiment_name'])
plan.manifest


Use `GOLD_INFERENCE` when features are already prepared; no source refinement or merge nodes are generated. The input must be unlabeled and include a unique nonblank `request_id`. Data assets are explicitly registered after a successful job with `project.register_outputs(plan, job_name)`; `project.get_dataset(name, version)` requires the same factory/project/environment.

For Azure Data Factory, the server's `DataOpsAdapter` accepts the same three parameters regardless of dataset count. It resolves a fresh graph and concrete lake paths for each UTC date and model. A 202 response is acceptance, not completion; poll job status and fail on failed/cancelled/unknown outcomes. The old SDK v1 ExecutePipeline activity is not used.

In [ ]:
adf_request = {'esml_data_date_utc': request.data_date_utc, 'esml_model_version': request.model_version, 'esml_run_id': request.run_id}
adf_request  # Preview only; no HTTP request or cloud invocation.


A published pipeline-component batch deployment is bound to its reviewed plan and graph. Rebinding it with a different date/model while retaining old embedded configuration is refused; rebuild and submit through the adapter, or explicitly publish a new version. Private connectivity, identities, dataset onboarding and deployment approvals remain separate prerequisites.